In [1]:
from model import aasist3


model = aasist3.from_pretrained(
    "./weight",
    local_files_only=True
)

model.eval()

b:\download\TruthSeeker-app\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights from local directory


aasist3(
  (w2v_encoder): Wav2Vec2Encoder(
    (model): Wav2Vec2Model(
      (feature_extractor): Wav2Vec2FeatureEncoder(
        (conv_layers): ModuleList(
          (0): Wav2Vec2LayerNormConvLayer(
            (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
            (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (activation): GELUActivation()
          )
          (1-4): 4 x Wav2Vec2LayerNormConvLayer(
            (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
            (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (activation): GELUActivation()
          )
          (5-6): 2 x Wav2Vec2LayerNormConvLayer(
            (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
            (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (activation): GELUActivation()
          )
        )
      )
      (feature_projection): Wav2Vec2FeatureProjection(
        (layer_norm):

In [12]:
import torch
import torchaudio

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NB_SAMP = 64600   # from config

def load_audio(path, target_sr=16000):
    wav, sr = torchaudio.load(path)

    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)

    wav = wav.mean(0)  # mono
    wav = wav / (wav.abs().max() + 1e-9)  # normalize
    return wav


def pad_or_trim(wav, length=NB_SAMP):
    if len(wav) > length:
        return wav[:length]
    elif len(wav) < length:
        pad = length - len(wav)
        return torch.nn.functional.pad(wav, (0, pad))
    return wav


In [6]:
mel_extractor = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_fft=512,
    win_length=400,
    hop_length=160,
    n_mels=80
).to(DEVICE)

def extract_features(wav):
    """
    wav: 1D tensor (T,)
    returns: (1, 80, T')
    """
    with torch.no_grad():
        mel = mel_extractor(wav.unsqueeze(0))

        # Handle both torchaudio behaviors safely
        if mel.dim() == 4:
            # (B, C, F, T) → (B, F, T)
            mel = mel[:, 0, :, :]
        elif mel.dim() == 3:
            # (B, F, T) → already correct
            pass
        else:
            raise RuntimeError(f"Unexpected mel shape: {mel.shape}")

        mel = torch.log(mel + 1e-6)

    return mel



In [13]:
import numpy as np

def detect_audio(audio_path, hop_sec=2.0):
    wav = load_audio(audio_path)
    sr = 16000

    hop_len = int(hop_sec * sr)
    scores = []

    for start in range(0, len(wav) - NB_SAMP + 1, hop_len):
        chunk = wav[start:start + NB_SAMP]
        chunk = pad_or_trim(chunk).to(DEVICE)

        with torch.no_grad():
            # 🔑 RAW waveform input
            logits = model(chunk.unsqueeze(0))  # (1, T)
            probs = torch.softmax(logits, dim=-1)
            scores.append(probs[0].cpu().numpy())

    if len(scores) == 0:
        return {"error": "Audio too short"}

    scores = np.array(scores).mean(axis=0)

    spoof = float(scores[0])
    bonafide = float(scores[1])

    if spoof > 0.65:
        label = "spoof"
    elif spoof < 0.4:
        label = "bonafide"
    else:
        label = "uncertain"

    return {
        "label": label,
        "spoof": spoof,
        "bonafide": bonafide
    }


In [19]:
result = detect_audio("real.mp3")
print(result)


{'label': 'spoof', 'spoof': 0.9999046325683594, 'bonafide': 9.548780508339405e-05}


In [5]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (PyTorch):", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


PyTorch version: 2.7.1+cu118
CUDA available: True
CUDA version (PyTorch): 11.8
GPU count: 1
GPU name: NVIDIA GeForce RTX 3060


In [11]:
from torchcodec.decoders import VideoDecoder;
  print('torchcodec works!')

IndentationError: unexpected indent (875507753.py, line 2)